In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib

In [2]:
df_he = pd.read_csv("HistogramEqualized.csv")
df_ah = pd.read_csv("AdaptiveHistogram.csv")

In [3]:
le = LabelEncoder()
y = le.fit_transform(df_he['Label'])  # Now y is numeric (0, 1)
X_he = df_he.drop(columns=['Label'])
X_ah = df_ah.drop(columns=['Label'])

In [4]:
combined_X = pd.concat([X_he, X_ah], axis=1)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(combined_X, y, test_size=0.2, random_state=42)

In [6]:
rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.9)),
    ("rf", RandomForestClassifier())
])

In [7]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.9)),
    ("knn", KNeighborsClassifier())
])

In [8]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.9)),
    ("svm", SVC(probability=True))
])

In [9]:
from sklearn.ensemble import VotingClassifier
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf_pipeline),
        ('knn', knn_pipeline),
        ('svm', svm_pipeline)
    ],
    voting='soft'
)

In [10]:
voting_clf.fit(X_train, y_train)
preds = voting_clf.predict(X_test)

In [11]:
print("Accuracy:", accuracy_score(y_test, preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

Accuracy: 0.9984321103794292
Confusion Matrix:
 [[1566    3]
 [   2 1618]]


In [12]:
joblib.dump(voting_clf, "soft_voting_model.pkl")

['soft_voting_model.pkl']

In [13]:
joblib.dump(le, 'label_encoder.pkl')

['label_encoder.pkl']